In [54]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Environment ready")

Environment ready


In [55]:
from pathlib import Path

project_path = Path.cwd().parent
dataset_path = project_path / "Dataset"

files = sorted(dataset_path.glob("*.csv"))

print("CSV files found:", len(files))

for file in files:
    print(file.name)

CSV files found: 18
HMRC_headcount_and_payroll_corrected_data_for_August_2025.csv
HMRC_headcount_and_payroll_data_for_April_2025.csv
HMRC_headcount_and_payroll_data_for_April_2026.csv
HMRC_headcount_and_payroll_data_for_December_2025.csv
HMRC_headcount_and_payroll_data_for_February_2025.csv
HMRC_headcount_and_payroll_data_for_February_2026.csv
HMRC_headcount_and_payroll_data_for_January_2025.csv
HMRC_headcount_and_payroll_data_for_January_2026.csv
HMRC_headcount_and_payroll_data_for_July_2025.csv
HMRC_headcount_and_payroll_data_for_June_2025.csv
HMRC_headcount_and_payroll_data_for_June_2026.csv
HMRC_headcount_and_payroll_data_for_March_2025.csv
HMRC_headcount_and_payroll_data_for_March_2026.csv
HMRC_headcount_and_payroll_data_for_May_2025.csv
HMRC_headcount_and_payroll_data_for_May_2026.csv
HMRC_headcount_and_payroll_data_for_November_2025.csv
HMRC_headcount_and_payroll_data_for_October_2025.csv
HMRC_headcount_and_payroll_data_for_September_2025.csv


In [56]:
monthly_data = []

for file in files:
    df = pd.read_csv(file)

    # Keep the source filename for validation and auditability
    df["source_file"] = file.name

    monthly_data.append(df)

master_df = pd.concat(
    monthly_data,
    ignore_index=True
)

print("Master dataset shape:", master_df.shape)


Master dataset shape: (33, 42)


In [57]:
print("Rows:", master_df.shape[0])
print("Columns:", master_df.shape[1])

display(
    pd.DataFrame({
        "Column": master_df.columns,
        "Data Type": master_df.dtypes.astype(str),
        "Missing Values": master_df.isna().sum().values
    })
)

Rows: 33
Columns: 42


,Column,Data Type,Missing Values
Year,Year,int64,0
Month,Month,str,0
Organisation name,Organisation name,str,0
Organisation type,Organisation type,str,0
"Main, parent or sponsoring department:","Main, parent or sponsoring department:",str,0
Payroll staff;\nAO/AA;\nHeadcount,Payroll staff;\nAO/AA;\nHeadcount,float64,0
Payroll staff;\nAO/AA;\nFull-time equivalent,Payroll staff;\nAO/AA;\nFull-time equivalent,float64,0
Payroll staff;\nEO;\nHeadcount,Payroll staff;\nEO;\nHeadcount,float64,0
Payroll staff;\nEO;\nFull-time equivalent,Payroll staff;\nEO;\nFull-time equivalent,float64,0
Payroll staff;\nSEO/HEO;\nHeadcount,Payroll staff;\nSEO/HEO;\nHeadcount,float64,0


In [58]:
monthly_check = (
    master_df
    .groupby(["Year", "Month"])
    .size()
    .reset_index(name="Rows")
)

display(monthly_check)

,Year,Month,Rows
0,2025,April,2
1,2025,August,2
2,2025,December,2
3,2025,February,2
4,2025,January,2
5,2025,July,2
6,2025,June,2
7,2025,March,2
8,2025,May,2
9,2025,November,2


In [59]:
# ==========================================
# DATA QUALITY ASSESSMENT
# ==========================================

print("Master dataset shape:", master_df.shape)

print("\nColumn count:", len(master_df.columns))

print("\nColumn names:")
for i, col in enumerate(master_df.columns, start=1):
    print(f"{i}. {col}")

print("\nDuplicate rows:", master_df.duplicated().sum())

print("\nMissing values:")
missing = (
    master_df.isna()
    .sum()
    .sort_values(ascending=False)
)

display(missing[missing > 0].to_frame("Missing_Count"))

print("\nData types:")
display(master_df.dtypes.to_frame("Data_Type"))

print("\nOrganisation counts by month:")
display(
    master_df.groupby(["Year", "Month"])["Organisation"]
    .nunique()
    .reset_index(name="Organisation_Count")
)

Master dataset shape: (33, 42)

Column count: 42

Column names:
1. Year
2. Month
3. Organisation name
4. Organisation type
5. Main, parent or sponsoring department:
6. Payroll staff;
AO/AA;
Headcount
7. Payroll staff;
AO/AA;
Full-time equivalent
8. Payroll staff;
EO;
Headcount
9. Payroll staff;
EO;
Full-time equivalent
10. Payroll staff;
SEO/HEO;
Headcount
11. Payroll staff;
SEO/HEO;
Full-time equivalent
12. Payroll staff;
Grade 6/7;
Headcount
13. Payroll staff;
Grade 6/7;
Full-time equivalent
14. Payroll staff;
SCS;
Headcount
15. Payroll staff;
SCS;
Full-time equivalent
16. Payroll staff;
Other, unknown, unspecified;
Headcount
17. Payroll staff;
Other, unknown, unspecified;
Full-time equivalent
18. Payroll staff;
Total;
Headcount
19. Payroll staff;
Total;
Full-time equivalent
20. Non-payroll staff;
Admin and Clerical;
Headcount
21. Non-payroll staff;
Admin and Clerical;
Full-time equivalent
22. Non-payroll staff;
Interim Managers & Specialist Contractors & Medical;
Headcount
23. Non-p

,Missing_Count
"Payroll staff;\nOther, unknown, unspecified;\nFull-time equivalent",1
"Payroll staff;\nOther, unknown, unspecified;\nHeadcount",1



Data types:


,Data_Type
Year,int64
Month,str
Organisation name,str
Organisation type,str
"Main, parent or sponsoring department:",str
Payroll staff;\nAO/AA;\nHeadcount,float64
Payroll staff;\nAO/AA;\nFull-time equivalent,float64
Payroll staff;\nEO;\nHeadcount,float64
Payroll staff;\nEO;\nFull-time equivalent,float64
Payroll staff;\nSEO/HEO;\nHeadcount,float64



Organisation counts by month:


KeyError: 'Column not found: Organisation'

In [62]:
# Clean column names safely
master_df.columns = (
    master_df.columns
    .astype(str)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

print("Organisation columns found:")

for col in master_df.columns:
    if "organisation" in col.lower():
        print(repr(col))

Organisation columns found:
'Organisation name'
'Organisation type'


In [61]:
# Identify the actual organisation column
org_col = next(
    col for col in master_df.columns
    if "organisation" in col.lower()
    and "parent" not in col.lower()
)

print("Using organisation column:", repr(org_col))

organisation_check = (
    master_df
    .groupby(["Year", "Month"])[org_col]
    .nunique()
    .reset_index(name="Organisation_Count")
)

display(organisation_check)

Using organisation column: 'Organisation name'


,Year,Month,Organisation_Count
0,2025,April,2
1,2025,August,2
2,2025,December,2
3,2025,February,2
4,2025,January,2
5,2025,July,2
6,2025,June,2
7,2025,March,2
8,2025,May,2
9,2025,November,2


In [ ]:
# ==========================================
# SCHEMA CONSISTENCY CHECK
# ==========================================

schema_check = []

for file in files:
    temp = pd.read_csv(file)

    schema_check.append({
        "file": file.name,
        "rows": len(temp),
        "columns": len(temp.columns),
        "column_signature": tuple(temp.columns)
    })

schema_df = pd.DataFrame(schema_check)

display(
    schema_df[
        ["file", "rows", "columns"]
    ]
)

print(
    "\nUnique column structures:",
    schema_df["column_signature"].nunique()
)

,file,rows,columns
0,HMRC_headcount_and_payroll_corrected_data_for_...,2,41
1,HMRC_headcount_and_payroll_data_for_April_2025...,2,41
2,HMRC_headcount_and_payroll_data_for_April_2026...,1,41
3,HMRC_headcount_and_payroll_data_for_December_2...,2,41
4,HMRC_headcount_and_payroll_data_for_February_2...,2,41
5,HMRC_headcount_and_payroll_data_for_February_2...,2,41
6,HMRC_headcount_and_payroll_data_for_January_20...,2,41
7,HMRC_headcount_and_payroll_data_for_January_20...,2,41
8,HMRC_headcount_and_payroll_data_for_July_2025.csv,2,41
9,HMRC_headcount_and_payroll_data_for_June_2025.csv,2,41



Unique column structures: 1


In [60]:
["Organisation"]

['Organisation']

In [63]:
organisation_check = (
    master_df
    .groupby(["Year", "Month"])["Organisation"]
    .nunique()
    .reset_index(name="Organisation_Count")
)

display(organisation_check)

KeyError: 'Column not found: Organisation'

In [64]:
for i, col in enumerate(master_df.columns, start=1):
    print(f"{i}: {col}")

1: Year
2: Month
3: Organisation name
4: Organisation type
5: Main, parent or sponsoring department:
6: Payroll staff; AO/AA; Headcount
7: Payroll staff; AO/AA; Full-time equivalent
8: Payroll staff; EO; Headcount
9: Payroll staff; EO; Full-time equivalent
10: Payroll staff; SEO/HEO; Headcount
11: Payroll staff; SEO/HEO; Full-time equivalent
12: Payroll staff; Grade 6/7; Headcount
13: Payroll staff; Grade 6/7; Full-time equivalent
14: Payroll staff; SCS; Headcount
15: Payroll staff; SCS; Full-time equivalent
16: Payroll staff; Other, unknown, unspecified; Headcount
17: Payroll staff; Other, unknown, unspecified; Full-time equivalent
18: Payroll staff; Total; Headcount
19: Payroll staff; Total; Full-time equivalent
20: Non-payroll staff; Admin and Clerical; Headcount
21: Non-payroll staff; Admin and Clerical; Full-time equivalent
22: Non-payroll staff; Interim Managers & Specialist Contractors & Medical; Headcount
23: Non-payroll staff; Interim Managers & Specialist Contractors & Medica

In [65]:
# ==========================================
# ORGANISATION STRUCTURE CHECK
# ==========================================

# Detect organisation-related columns
org_name_col = next(
    col for col in master_df.columns
    if col.lower().strip() == "organisation name"
)

org_type_col = next(
    col for col in master_df.columns
    if col.lower().strip() == "organisation type"
)

print("Organisation name column:", org_name_col)
print("Organisation type column:", org_type_col)

# 1. Organisations in the dataset
print("\nOrganisation summary:")

organisation_summary = (
    master_df[org_name_col]
    .dropna()
    .astype(str)
    .str.strip()
    .value_counts()
    .rename_axis("Organisation")
    .reset_index(name="Rows")
)

display(organisation_summary)

# 2. Organisation type summary
print("\nOrganisation type summary:")

organisation_type_summary = (
    master_df[org_type_col]
    .dropna()
    .astype(str)
    .str.strip()
    .value_counts()
    .rename_axis("Organisation_Type")
    .reset_index(name="Rows")
)

display(organisation_type_summary)

# 3. Monthly organisation count
print("\nMonthly organisation count:")

monthly_org = (
    master_df
    .groupby(["Year", "Month"])
    .agg(
        Organisation_Count=(org_name_col, "nunique")
    )
    .reset_index()
)

display(monthly_org)

Organisation name column: Organisation name
Organisation type column: Organisation type

Organisation summary:


,Organisation,Rows
0,HM Revenue and Customs,18
1,Valuation Office Agency,14
2,Valuation Office,1



Organisation type summary:


,Organisation_Type,Rows
0,Non-Ministerial Department,18
1,Executive Agency,15



Monthly organisation count:


,Year,Month,Organisation_Count
0,2025,April,2
1,2025,August,2
2,2025,December,2
3,2025,February,2
4,2025,January,2
5,2025,July,2
6,2025,June,2
7,2025,March,2
8,2025,May,2
9,2025,November,2


In [66]:
# ==========================================
# STANDARDISE COLUMN NAMES
# ==========================================

def clean_column_name(col):
    col = str(col).strip().lower()
    col = col.replace("&", "and")
    col = col.replace("/", "_")
    col = col.replace("-", "_")
    col = col.replace(";", "_")
    col = col.replace(",", "")
    col = col.replace("(", "")
    col = col.replace(")", "")
    col = "_".join(col.split())
    while "__" in col:
        col = col.replace("__", "_")
    return col.strip("_")


master_df.columns = [clean_column_name(col) for col in master_df.columns]

print("Standardised columns:")
for i, col in enumerate(master_df.columns, start=1):
    print(f"{i}: {col}")

Standardised columns:
1: year
2: month
3: organisation_name
4: organisation_type
5: main_parent_or_sponsoring_department:
6: payroll_staff_ao_aa_headcount
7: payroll_staff_ao_aa_full_time_equivalent
8: payroll_staff_eo_headcount
9: payroll_staff_eo_full_time_equivalent
10: payroll_staff_seo_heo_headcount
11: payroll_staff_seo_heo_full_time_equivalent
12: payroll_staff_grade_6_7_headcount
13: payroll_staff_grade_6_7_full_time_equivalent
14: payroll_staff_scs_headcount
15: payroll_staff_scs_full_time_equivalent
16: payroll_staff_other_unknown_unspecified_headcount
17: payroll_staff_other_unknown_unspecified_full_time_equivalent
18: payroll_staff_total_headcount
19: payroll_staff_total_full_time_equivalent
20: non_payroll_staff_admin_and_clerical_headcount
21: non_payroll_staff_admin_and_clerical_full_time_equivalent
22: non_payroll_staff_interim_managers_and_specialist_contractors_and_medical_headcount
23: non_payroll_staff_interim_managers_and_specialist_contractors_and_medical_full_tim

In [67]:
# ==========================================
# IDENTIFY CORE BUSINESS METRICS
# ==========================================

def find_columns(keyword_groups):
    matches = []

    for col in master_df.columns:
        col_lower = col.lower()

        if all(
            any(keyword in col_lower for keyword in group)
            for group in keyword_groups
        ):
            matches.append(col)

    return matches


print("Potential TOTAL PAYROLL columns:")
print(
    find_columns([
        ["payroll"],
        ["total"],
        ["headcount", "full_time_equivalent"]
    ])
)

print("\nPotential GRAND TOTAL COST columns:")
print(
    find_columns([
        ["grand", "total"],
        ["paybill", "staffing"],
        ["cost"]
    ])
)

print("\nPotential PAYROLL COST columns:")
print(
    find_columns([
        ["payroll"],
        ["cost"]
    ])
)

print("\nPotential NON-PAYROLL COST columns:")
print(
    find_columns([
        ["non_payroll", "non"],
        ["cost"]
    ])
)

Potential TOTAL PAYROLL columns:
['payroll_staff_total_headcount', 'payroll_staff_total_full_time_equivalent', 'non_payroll_staff_total_contingent_labour_headcount', 'non_payroll_staff_total_contingent_labour_full_time_equivalent']

Potential GRAND TOTAL COST columns:
['payroll_staff_costs_total_paybill', 'grand_total_paybill_staffing_payroll_and_non_payroll_costs']

Potential PAYROLL COST columns:
['payroll_staff_costs_salary', 'payroll_staff_costs_allowances', 'payroll_staff_costs_non_consolidated_performance_payments', 'payroll_staff_costs_overtime', 'payroll_staff_costs_employer_pension_contributions', 'payroll_staff_costs_employer_national_insurance_contributions', 'payroll_staff_costs_total_paybill', 'non_payroll_staff_costs_contingent_labour', 'non_payroll_staff_costs_consultancy', 'non_payroll_staff_costs_total_staff_costs', 'grand_total_paybill_staffing_payroll_and_non_payroll_costs']

Potential NON-PAYROLL COST columns:
['payroll_staff_costs_non_consolidated_performance_payme

In [69]:
# ==========================================
# CORE WORKFORCE & COST KPI DATASET
# ==========================================

kpi_df = master_df.copy()

# Convert key numeric fields
numeric_columns = [
    "payroll_staff_total_headcount",
    "payroll_staff_total_full_time_equivalent",
    "payroll_staff_costs_total_paybill",
    "grand_total_paybill_staffing_payroll_and_non_payroll_costs"
]

for col in numeric_columns:
    if col in kpi_df.columns:
        kpi_df[col] = pd.to_numeric(
            kpi_df[col],
            errors="coerce"
        )

# Core metrics
kpi_df["total_workforce"] = kpi_df[
    "payroll_staff_total_headcount"
]

kpi_df["total_fte"] = kpi_df[
    "payroll_staff_total_full_time_equivalent"
]

kpi_df["total_staffing_cost"] = kpi_df[
    "grand_total_paybill_staffing_payroll_and_non_payroll_costs"
]

# Cost per FTE
kpi_df["cost_per_fte"] = np.where(
    kpi_df["total_fte"] > 0,
    kpi_df["total_staffing_cost"] / kpi_df["total_fte"],
    np.nan
)

# FTE / Headcount ratio
kpi_df["fte_to_headcount_ratio"] = np.where(
    kpi_df["total_workforce"] > 0,
    kpi_df["total_fte"] / kpi_df["total_workforce"],
    np.nan
)

# Display
core_columns = [
    "year",
    "month",
    "organisation_name",
    "organisation_type",
    "total_workforce",
    "total_fte",
    "total_staffing_cost",
    "cost_per_fte",
    "fte_to_headcount_ratio"
]

display(kpi_df[core_columns].head(20))

,year,month,organisation_name,organisation_type,total_workforce,total_fte,total_staffing_cost,cost_per_fte,fte_to_headcount_ratio
0,2025,August,HM Revenue and Customs,Non-Ministerial Department,NaN,NaN,NaN,NaN,NaN
1,2025,August,Valuation Office Agency,Executive Agency,NaN,NaN,NaN,NaN,NaN
2,2025,April,HM Revenue and Customs,Non-Ministerial Department,NaN,NaN,NaN,NaN,NaN
3,2025,April,Valuation Office Agency,Executive Agency,NaN,NaN,NaN,NaN,NaN
4,2026,April,HM Revenue and Customs,Non-Ministerial Department,NaN,NaN,NaN,NaN,NaN
5,2025,December,HM Revenue and Customs,Non-Ministerial Department,NaN,NaN,NaN,NaN,NaN
6,2025,December,Valuation Office Agency,Executive Agency,NaN,NaN,NaN,NaN,NaN
7,2025,February,HM Revenue and Customs,Non-Ministerial Department,66440.0,61728.00,2.960169e+08,4795.504973,0.929079
8,2025,February,Valuation Office Agency,Executive Agency,3980.0,3730.00,1.786567e+07,4789.723324,0.937186
9,2026,February,HM Revenue and Customs,Non-Ministerial Department,NaN,NaN,NaN,NaN,NaN


In [70]:
# ==========================================
# KPI VALIDATION
# ==========================================

print("Total rows:", len(kpi_df))

print("\nMissing KPI values:")
display(
    kpi_df[
        [
            "total_workforce",
            "total_fte",
            "total_staffing_cost",
            "cost_per_fte"
        ]
    ]
    .isna()
    .sum()
    .to_frame("Missing_Count")
)

print("\nKPI summary:")
display(
    kpi_df[
        [
            "total_workforce",
            "total_fte",
            "total_staffing_cost",
            "cost_per_fte",
            "fte_to_headcount_ratio"
        ]
    ].describe()
)

Total rows: 33

Missing KPI values:


,Missing_Count
total_workforce,28
total_fte,28
total_staffing_cost,28
cost_per_fte,28



KPI summary:


,total_workforce,total_fte,total_staffing_cost,cost_per_fte,fte_to_headcount_ratio
count,5.000000,5.000000,5.000000e+00,5.000000,5.000000
mean,43600.400000,40608.784000,1.980644e+08,4833.697462,0.933667
std,36294.125176,33793.724369,1.658523e+08,100.052541,0.004036
min,3980.000000,3730.000000,1.786567e+07,4751.662785,0.929079
25%,4108.000000,3853.630000,1.831115e+07,4789.723324,0.930171
50%,66440.000000,61728.000000,2.960169e+08,4795.504973,0.933820
75%,67575.000000,62856.290000,3.032902e+08,4825.136845,0.937186
max,75899.000000,70876.000000,3.548378e+08,5006.459380,0.938079


In [71]:
# ==========================================
# FIX NUMERIC / CURRENCY VALUES
# ==========================================

def clean_numeric(series):
    return pd.to_numeric(
        series.astype(str)
        .str.replace("£", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.replace("%", "", regex=False)
        .str.strip()
        .replace({"": np.nan, "nan": np.nan, "None": np.nan}),
        errors="coerce"
    )


numeric_columns = [
    "payroll_staff_total_headcount",
    "payroll_staff_total_full_time_equivalent",
    "payroll_staff_costs_total_paybill",
    "grand_total_paybill_staffing_payroll_and_non_payroll_costs"
]

for col in numeric_columns:
    kpi_df[col] = clean_numeric(kpi_df[col])


# Recalculate KPI fields
kpi_df["total_workforce"] = kpi_df[
    "payroll_staff_total_headcount"
]

kpi_df["total_fte"] = kpi_df[
    "payroll_staff_total_full_time_equivalent"
]

kpi_df["total_staffing_cost"] = kpi_df[
    "grand_total_paybill_staffing_payroll_and_non_payroll_costs"
]

kpi_df["cost_per_fte"] = np.where(
    kpi_df["total_fte"] > 0,
    kpi_df["total_staffing_cost"] / kpi_df["total_fte"],
    np.nan
)

kpi_df["fte_to_headcount_ratio"] = np.where(
    kpi_df["total_workforce"] > 0,
    kpi_df["total_fte"] / kpi_df["total_workforce"],
    np.nan
)

print("Numeric conversion completed.")

Numeric conversion completed.


In [72]:
# ==========================================
# VALIDATE KPI DATA
# ==========================================

validation = kpi_df[
    [
        "year",
        "month",
        "organisation_name",
        "total_workforce",
        "total_fte",
        "total_staffing_cost",
        "cost_per_fte",
        "fte_to_headcount_ratio"
    ]
].copy()

display(validation.head(20))

print("\nValid workforce rows:",
      kpi_df["total_workforce"].notna().sum())

print("Valid FTE rows:",
      kpi_df["total_fte"].notna().sum())

print("Valid staffing-cost rows:",
      kpi_df["total_staffing_cost"].notna().sum())

,year,month,organisation_name,total_workforce,total_fte,total_staffing_cost,cost_per_fte,fte_to_headcount_ratio
0,2025,August,HM Revenue and Customs,NaN,NaN,NaN,NaN,NaN
1,2025,August,Valuation Office Agency,NaN,NaN,NaN,NaN,NaN
2,2025,April,HM Revenue and Customs,NaN,NaN,NaN,NaN,NaN
3,2025,April,Valuation Office Agency,NaN,NaN,NaN,NaN,NaN
4,2026,April,HM Revenue and Customs,NaN,NaN,NaN,NaN,NaN
5,2025,December,HM Revenue and Customs,NaN,NaN,NaN,NaN,NaN
6,2025,December,Valuation Office Agency,NaN,NaN,NaN,NaN,NaN
7,2025,February,HM Revenue and Customs,66440.0,61728.00,2.960169e+08,4795.504973,0.929079
8,2025,February,Valuation Office Agency,3980.0,3730.00,1.786567e+07,4789.723324,0.937186
9,2026,February,HM Revenue and Customs,NaN,NaN,NaN,NaN,NaN



Valid workforce rows: 5
Valid FTE rows: 5
Valid staffing-cost rows: 5


In [73]:
# ==========================================
# DIAGNOSE NUMERIC CONVERSION PROBLEM
# ==========================================

check_cols = [
    "payroll_staff_total_headcount",
    "payroll_staff_total_full_time_equivalent",
    "grand_total_paybill_staffing_payroll_and_non_payroll_costs"
]

for col in check_cols:
    print("\n" + "=" * 70)
    print("COLUMN:", col)

    # Show raw values exactly as imported
    raw_values = master_df[col].dropna().astype(str)

    print("\nSample raw values:")
    for value in raw_values.head(15):
        print(repr(value))

    # Values that fail numeric conversion
    converted = pd.to_numeric(
        raw_values
        .str.replace("£", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.strip(),
        errors="coerce"
    )

    bad_values = raw_values[converted.isna()].drop_duplicates()

    print("\nValues that fail conversion:")
    for value in bad_values.head(20):
        print(repr(value))


COLUMN: payroll_staff_total_headcount

Sample raw values:
'67,751'
'4,072'
'66,965'
'3,975'
'75,493'
'70,061'
'4,263'
'66440'
'3980'
'70,744'
'4,306'
'66,112'
'3,977'
'70,519'
'4,251'

Values that fail conversion:

COLUMN: payroll_staff_total_full_time_equivalent

Sample raw values:
'63,061'
'3,819'
'62,248'
'3,723'
'70,456'
'65,308'
'4,004'
'61728'
'3730'
'65,964'
'4,047'
'61,422'
'3,728'
'65,743'
'3,991'

Values that fail conversion:

COLUMN: grand_total_paybill_staffing_payroll_and_non_payroll_costs

Sample raw values:
'£305,862,378.75'
'£19,187,837.98'
'£302,977,587.27'
'£18,092,324.77'
'£354,700,225.41'
'£332,901,100.06'
'£20,486,407.13'
'296016931'
'17865668'
'£335,382,741.29'
'£20,593,826.33'
'£304,085,785.84'
'£18,017,126.19'
'£330,942,854.45'
'£20,307,580.86'

Values that fail conversion:


In [74]:
# ==========================================
# RAW CSV STRUCTURE CHECK
# ==========================================

import csv

test_file = dataset_path / "HMRC_headcount_and_payroll_data_for_January_2025.csv"

with open(test_file, "r", encoding="utf-8-sig", newline="") as f:
    reader = csv.reader(f)

    header = next(reader)
    first_row = next(reader)

print("Header columns:", len(header))
print("First row columns:", len(first_row))

print("\nHEADER:")
for i, value in enumerate(header, start=1):
    print(i, repr(value))

print("\nFIRST ROW:")
for i, value in enumerate(first_row, start=1):
    print(i, repr(value))

Header columns: 41
First row columns: 41

HEADER:
1 'Year'
2 'Month'
3 'Organisation name'
4 'Organisation type'
5 'Main, parent or sponsoring department:'
6 'Payroll staff;\nAO/AA;\nHeadcount'
7 'Payroll staff;\nAO/AA;\nFull-time equivalent'
8 'Payroll staff;\nEO;\nHeadcount'
9 'Payroll staff;\nEO;\nFull-time equivalent'
10 'Payroll staff;\nSEO/HEO;\nHeadcount'
11 'Payroll staff;\nSEO/HEO;\nFull-time equivalent'
12 'Payroll staff;\nGrade 6/7;\nHeadcount'
13 'Payroll staff;\nGrade 6/7;\nFull-time equivalent'
14 'Payroll staff;\nSCS;\nHeadcount'
15 'Payroll staff;\nSCS;\nFull-time equivalent'
16 'Payroll staff;\nOther, unknown, unspecified;\nHeadcount'
17 'Payroll staff;\nOther, unknown, unspecified;\nFull-time equivalent'
18 'Payroll staff;\nTotal;\nHeadcount'
19 'Payroll staff;\nTotal;\nFull-time equivalent'
20 'Non-payroll staff;\nAdmin and Clerical;\nHeadcount'
21 'Non-payroll staff;\nAdmin and Clerical;\nFull-time equivalent'
22 'Non-payroll staff;\nInterim Managers & Specialist Co

In [75]:
# ==========================================
# RAW HEADER + FIRST DATA ROW ALIGNMENT
# ==========================================

import csv

test_file = dataset_path / "HMRC_headcount_and_payroll_data_for_January_2025.csv"

with open(test_file, "r", encoding="utf-8-sig", newline="") as f:
    reader = csv.reader(f)
    header = next(reader)
    first_row = next(reader)

print("Header:", len(header))
print("First row:", len(first_row))

print("\nPOSITIONAL CHECK:")
for i, (h, v) in enumerate(zip(header, first_row), start=1):
    print(f"{i:02d} | HEADER: {h!r}")
    print(f"   | VALUE : {v!r}")

Header: 41
First row: 41

POSITIONAL CHECK:
01 | HEADER: 'Year'
   | VALUE : '2025'
02 | HEADER: 'Month'
   | VALUE : 'January'
03 | HEADER: 'Organisation name'
   | VALUE : 'HM Revenue and Customs'
04 | HEADER: 'Organisation type'
   | VALUE : 'Non-Ministerial Department'
05 | HEADER: 'Main, parent or sponsoring department:'
   | VALUE : 'HM Revenue and Customs'
06 | HEADER: 'Payroll staff;\nAO/AA;\nHeadcount'
   | VALUE : '17637'
07 | HEADER: 'Payroll staff;\nAO/AA;\nFull-time equivalent'
   | VALUE : '15527.8'
08 | HEADER: 'Payroll staff;\nEO;\nHeadcount'
   | VALUE : '14723'
09 | HEADER: 'Payroll staff;\nEO;\nFull-time equivalent'
   | VALUE : '13757.83'
10 | HEADER: 'Payroll staff;\nSEO/HEO;\nHeadcount'
   | VALUE : '24179'
11 | HEADER: 'Payroll staff;\nSEO/HEO;\nFull-time equivalent'
   | VALUE : '23011.39'
12 | HEADER: 'Payroll staff;\nGrade 6/7;\nHeadcount'
   | VALUE : '9057'
13 | HEADER: 'Payroll staff;\nGrade 6/7;\nFull-time equivalent'
   | VALUE : '8624.87'
14 | HEADER: 'P

In [76]:
# ==========================================
# ROBUST HMRC CSV IMPORT
# ==========================================

import csv
import pandas as pd
import numpy as np

all_rows = []
master_header = None
file_validation = []

for file in sorted(dataset_path.glob("*.csv")):

    with open(file, "r", encoding="utf-8-sig", newline="") as f:
        reader = csv.reader(f)

        header = next(reader)
        rows = list(reader)

    # First file defines the master schema
    if master_header is None:
        master_header = header

    # Validate schema
    header_match = header == master_header

    # Validate row lengths
    row_lengths = sorted(set(len(row) for row in rows))

    file_validation.append({
        "file": file.name,
        "rows": len(rows),
        "columns": len(header),
        "header_match": header_match,
        "row_lengths": row_lengths
    })

    # Add source file for audit trail
    for row in rows:
        if len(row) == len(master_header):
            all_rows.append(row)

# Create clean master dataframe
master_df = pd.DataFrame(
    all_rows,
    columns=master_header
)

# Clean header names
master_df.columns = (
    master_df.columns
    .astype(str)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

print("Master dataset created successfully.")
print("Rows:", master_df.shape[0])
print("Columns:", master_df.shape[1])

print("\nFile validation:")
display(pd.DataFrame(file_validation))

Master dataset created successfully.
Rows: 33
Columns: 41

File validation:


,file,rows,columns,header_match,row_lengths
0,HMRC_headcount_and_payroll_corrected_data_for_...,2,41,True,[41]
1,HMRC_headcount_and_payroll_data_for_April_2025...,2,41,True,[41]
2,HMRC_headcount_and_payroll_data_for_April_2026...,1,41,True,[41]
3,HMRC_headcount_and_payroll_data_for_December_2...,2,41,True,[41]
4,HMRC_headcount_and_payroll_data_for_February_2...,2,41,True,[41]
5,HMRC_headcount_and_payroll_data_for_February_2...,2,41,True,[41]
6,HMRC_headcount_and_payroll_data_for_January_20...,2,41,True,[41]
7,HMRC_headcount_and_payroll_data_for_January_20...,2,41,True,[41]
8,HMRC_headcount_and_payroll_data_for_July_2025.csv,2,41,True,[41]
9,HMRC_headcount_and_payroll_data_for_June_2025.csv,2,41,True,[41]


In [77]:
# ==========================================
# VERIFY RAW IMPORT
# ==========================================

print("Expected columns:", 41)
print("Imported columns:", len(master_df.columns))

print("\nFirst row:")
display(master_df.head(2).T)

Expected columns: 41
Imported columns: 41

First row:


,0,1
Year,2025,2025
Month,August,August
Organisation name,HM Revenue and Customs,Valuation Office Agency
Organisation type,Non-Ministerial Department,Executive Agency
"Main, parent or sponsoring department:",HM Revenue and Customs,HM Revenue and Customs
Payroll staff; AO/AA; Headcount,18210,846
Payroll staff; AO/AA; Full-time equivalent,16155.03,783.78
Payroll staff; EO; Headcount,15159,900
Payroll staff; EO; Full-time equivalent,14204.33,835.38
Payroll staff; SEO/HEO; Headcount,24524,1759


In [78]:
# ==========================================
# VERIFY KEY BUSINESS FIELDS
# ==========================================

key_columns = [
    "Payroll staff;\nTotal;\nHeadcount",
    "Payroll staff;\nTotal;\nFull-time equivalent",
    "Payroll staff costs;\nTotal paybill",
    "Grand Total paybill/staffing (payroll and non-payroll) costs"
]

for col in key_columns:
    print("\nCOLUMN:", repr(col))
    print(master_df[col].head(5).tolist())


COLUMN: 'Payroll staff;\nTotal;\nHeadcount'


KeyError: 'Payroll staff;\nTotal;\nHeadcount'

In [79]:
# ==========================================
# FINALISE CLEAN COLUMN NAMES
# ==========================================

def clean_column_name(col):
    col = str(col).strip().lower()
    col = col.replace("&", "and")
    col = col.replace("/", "_")
    col = col.replace("-", "_")
    col = col.replace(";", "_")
    col = "_".join(col.split())
    
    while "__" in col:
        col = col.replace("__", "_")
        
    return col.strip("_")


master_df.columns = [
    clean_column_name(col)
    for col in master_df.columns
]

print("Cleaned columns:")
for i, col in enumerate(master_df.columns, start=1):
    print(f"{i}: {col}")

Cleaned columns:
1: year
2: month
3: organisation_name
4: organisation_type
5: main,_parent_or_sponsoring_department:
6: payroll_staff_ao_aa_headcount
7: payroll_staff_ao_aa_full_time_equivalent
8: payroll_staff_eo_headcount
9: payroll_staff_eo_full_time_equivalent
10: payroll_staff_seo_heo_headcount
11: payroll_staff_seo_heo_full_time_equivalent
12: payroll_staff_grade_6_7_headcount
13: payroll_staff_grade_6_7_full_time_equivalent
14: payroll_staff_scs_headcount
15: payroll_staff_scs_full_time_equivalent
16: payroll_staff_other,_unknown,_unspecified_headcount
17: payroll_staff_other,_unknown,_unspecified_full_time_equivalent
18: payroll_staff_total_headcount
19: payroll_staff_total_full_time_equivalent
20: non_payroll_staff_admin_and_clerical_headcount
21: non_payroll_staff_admin_and_clerical_full_time_equivalent
22: non_payroll_staff_interim_managers_and_specialist_contractors_and_medical_headcount
23: non_payroll_staff_interim_managers_and_specialist_contractors_and_medical_full_tim

In [80]:
# ==========================================
# IDENTIFY CORE COLUMNS
# ==========================================

def find_one(*keywords):
    matches = []

    for col in master_df.columns:
        col_lower = col.lower()

        if all(keyword.lower() in col_lower for keyword in keywords):
            matches.append(col)

    if len(matches) == 1:
        return matches[0]

    print(f"\nSearch {keywords} found:")
    for m in matches:
        print(" -", m)

    return matches[0] if matches else None


headcount_col = find_one("payroll_staff", "total", "headcount")
fte_col = find_one("payroll_staff", "total", "full_time_equivalent")
payroll_cost_col = find_one("payroll_staff_costs", "total_paybill")
grand_cost_col = find_one("grand_total", "paybill", "staffing")

print("\nSelected columns:")
print("Headcount:", headcount_col)
print("FTE:", fte_col)
print("Payroll cost:", payroll_cost_col)
print("Grand total cost:", grand_cost_col)


Search ('payroll_staff', 'total', 'headcount') found:
 - payroll_staff_total_headcount
 - non_payroll_staff_total_contingent_labour_headcount

Search ('payroll_staff', 'total', 'full_time_equivalent') found:
 - payroll_staff_total_full_time_equivalent
 - non_payroll_staff_total_contingent_labour_full_time_equivalent

Selected columns:
Headcount: payroll_staff_total_headcount
FTE: payroll_staff_total_full_time_equivalent
Payroll cost: payroll_staff_costs_total_paybill
Grand total cost: grand_total_paybill_staffing_(payroll_and_non_payroll)_costs


In [81]:
# ==========================================
# CORE KPI DATASET
# ==========================================

kpi_df = master_df.copy()

def clean_numeric(series):
    return pd.to_numeric(
        series.astype(str)
        .str.replace("£", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.strip(),
        errors="coerce"
    )


# Convert actual selected fields
kpi_df[headcount_col] = clean_numeric(kpi_df[headcount_col])
kpi_df[fte_col] = clean_numeric(kpi_df[fte_col])
kpi_df[payroll_cost_col] = clean_numeric(kpi_df[payroll_cost_col])
kpi_df[grand_cost_col] = clean_numeric(kpi_df[grand_cost_col])


# Create business-friendly KPI fields
kpi_df["total_workforce"] = kpi_df[headcount_col]
kpi_df["total_fte"] = kpi_df[fte_col]
kpi_df["payroll_cost"] = kpi_df[payroll_cost_col]
kpi_df["total_staffing_cost"] = kpi_df[grand_cost_col]

kpi_df["cost_per_fte"] = np.where(
    kpi_df["total_fte"] > 0,
    kpi_df["total_staffing_cost"] / kpi_df["total_fte"],
    np.nan
)

kpi_df["fte_to_headcount_ratio"] = np.where(
    kpi_df["total_workforce"] > 0,
    kpi_df["total_fte"] / kpi_df["total_workforce"],
    np.nan
)


display(
    kpi_df[
        [
            "year",
            "month",
            "organisation_name",
            "organisation_type",
            "total_workforce",
            "total_fte",
            "payroll_cost",
            "total_staffing_cost",
            "cost_per_fte",
            "fte_to_headcount_ratio"
        ]
    ].head(10)
)

,year,month,organisation_name,organisation_type,total_workforce,total_fte,payroll_cost,total_staffing_cost,cost_per_fte,fte_to_headcount_ratio
0,2025,August,HM Revenue and Customs,Non-Ministerial Department,67751.0,63061.0,2.983809e+08,3.058624e+08,4850.262107,0.930776
1,2025,August,Valuation Office Agency,Executive Agency,4072.0,3819.0,1.821046e+07,1.918784e+07,5024.309500,0.937868
2,2025,April,HM Revenue and Customs,Non-Ministerial Department,66965.0,62248.0,2.938939e+08,3.029776e+08,4867.266214,0.929560
3,2025,April,Valuation Office Agency,Executive Agency,3975.0,3723.0,1.778231e+07,1.809232e+07,4859.609124,0.936604
4,2026,April,HM Revenue and Customs,Non-Ministerial Department,75493.0,70456.0,3.427328e+08,3.547002e+08,5034.350877,0.933279
5,2025,December,HM Revenue and Customs,Non-Ministerial Department,70061.0,65308.0,3.228710e+08,3.329011e+08,5097.401544,0.932159
6,2025,December,Valuation Office Agency,Executive Agency,4263.0,4004.0,1.966084e+07,2.048641e+07,5116.485297,0.939245
7,2025,February,HM Revenue and Customs,Non-Ministerial Department,66440.0,61728.0,2.861316e+08,2.960169e+08,4795.504973,0.929079
8,2025,February,Valuation Office Agency,Executive Agency,3980.0,3730.0,1.760205e+07,1.786567e+07,4789.723324,0.937186
9,2026,February,HM Revenue and Customs,Non-Ministerial Department,70744.0,65964.0,3.230202e+08,3.353827e+08,5084.329957,0.932432


In [82]:
# ==========================================
# MONTHLY WORKFORCE & COST KPI SUMMARY
# ==========================================

# Convert month names into proper chronological order
month_order = [
    "January", "February", "March", "April",
    "May", "June", "July", "August",
    "September", "October", "November", "December"
]

kpi_df["month"] = pd.Categorical(
    kpi_df["month"],
    categories=month_order,
    ordered=True
)

monthly_kpi = (
    kpi_df
    .groupby(["year", "month"], observed=True)
    .agg(
        total_workforce=("total_workforce", "sum"),
        total_fte=("total_fte", "sum"),
        payroll_cost=("payroll_cost", "sum"),
        total_staffing_cost=("total_staffing_cost", "sum")
    )
    .reset_index()
    .sort_values(["year", "month"])
)

monthly_kpi["cost_per_fte"] = np.where(
    monthly_kpi["total_fte"] > 0,
    monthly_kpi["total_staffing_cost"] /
    monthly_kpi["total_fte"],
    np.nan
)

monthly_kpi["fte_to_headcount_ratio"] = np.where(
    monthly_kpi["total_workforce"] > 0,
    monthly_kpi["total_fte"] /
    monthly_kpi["total_workforce"],
    np.nan
)

display(monthly_kpi)

,year,month,total_workforce,total_fte,payroll_cost,total_staffing_cost,cost_per_fte,fte_to_headcount_ratio
0,2025,January,70089.0,65150.00,3.103612e+08,3.221029e+08,4944.020139,0.929532
1,2025,February,70420.0,65458.00,3.037337e+08,3.138826e+08,4795.175517,0.929537
2,2025,March,70929.0,65955.00,3.057393e+08,3.159755e+08,4790.774230,0.929874
3,2025,April,70940.0,65971.00,3.116762e+08,3.210699e+08,4866.834094,0.929955
4,2025,May,70897.0,65939.00,3.131183e+08,3.212196e+08,4871.465706,0.930068
5,2025,June,71683.0,66709.92,3.129913e+08,3.216014e+08,4820.892471,0.930624
6,2025,July,71820.0,66857.00,3.188418e+08,3.306860e+08,4946.168208,0.930897
7,2025,August,71823.0,66880.00,3.165914e+08,3.250502e+08,4860.200609,0.931178
8,2025,September,72886.0,67897.00,3.643523e+08,3.740170e+08,5508.594492,0.931551
9,2025,October,73340.0,68347.00,3.335693e+08,3.454291e+08,5054.048857,0.931920


In [83]:
# ==========================================
# MONTH-OVER-MONTH BUSINESS MOVEMENT
# ==========================================

monthly_kpi["workforce_change_pct"] = (
    monthly_kpi["total_workforce"]
    .pct_change() * 100
)

monthly_kpi["fte_change_pct"] = (
    monthly_kpi["total_fte"]
    .pct_change() * 100
)

monthly_kpi["staffing_cost_change_pct"] = (
    monthly_kpi["total_staffing_cost"]
    .pct_change() * 100
)

monthly_kpi["cost_per_fte_change_pct"] = (
    monthly_kpi["cost_per_fte"]
    .pct_change() * 100
)

display(monthly_kpi)

,year,month,total_workforce,total_fte,payroll_cost,total_staffing_cost,cost_per_fte,fte_to_headcount_ratio,workforce_change_pct,fte_change_pct,staffing_cost_change_pct,cost_per_fte_change_pct
0,2025,January,70089.0,65150.00,3.103612e+08,3.221029e+08,4944.020139,0.929532,NaN,NaN,NaN,NaN
1,2025,February,70420.0,65458.00,3.037337e+08,3.138826e+08,4795.175517,0.929537,0.472257,0.472755,-2.552077,-3.010599
2,2025,March,70929.0,65955.00,3.057393e+08,3.159755e+08,4790.774230,0.929874,0.722806,0.759265,0.666783,-0.091786
3,2025,April,70940.0,65971.00,3.116762e+08,3.210699e+08,4866.834094,0.929955,0.015508,0.024259,1.612276,1.587632
4,2025,May,70897.0,65939.00,3.131183e+08,3.212196e+08,4871.465706,0.930068,-0.060615,-0.048506,0.046615,0.095167
5,2025,June,71683.0,66709.92,3.129913e+08,3.216014e+08,4820.892471,0.930624,1.108651,1.169141,0.118851,-1.038152
6,2025,July,71820.0,66857.00,3.188418e+08,3.306860e+08,4946.168208,0.930897,0.191119,0.220477,2.824807,2.598601
7,2025,August,71823.0,66880.00,3.165914e+08,3.250502e+08,4860.200609,0.931178,0.004177,0.034402,-1.704261,-1.738065
8,2025,September,72886.0,67897.00,3.643523e+08,3.740170e+08,5508.594492,0.931551,1.480027,1.520634,15.064387,13.340887
9,2025,October,73340.0,68347.00,3.335693e+08,3.454291e+08,5054.048857,0.931920,0.622891,0.662769,-7.643492,-8.251572


In [84]:
# ==========================================
# BUSINESS EXCEPTION DETECTION
# ==========================================

exceptions = monthly_kpi[
    (
        monthly_kpi["staffing_cost_change_pct"].abs() >= 5
    )
    |
    (
        monthly_kpi["workforce_change_pct"].abs() >= 2
    )
    |
    (
        monthly_kpi["cost_per_fte_change_pct"].abs() >= 3
    )
].copy()

display(exceptions)

,year,month,total_workforce,total_fte,payroll_cost,total_staffing_cost,cost_per_fte,fte_to_headcount_ratio,workforce_change_pct,fte_change_pct,staffing_cost_change_pct,cost_per_fte_change_pct
1,2025,February,70420.0,65458.0,3.037337e+08,3.138826e+08,4795.175517,0.929537,0.472257,0.472755,-2.552077,-3.010599
8,2025,September,72886.0,67897.0,3.643523e+08,3.740170e+08,5508.594492,0.931551,1.480027,1.520634,15.064387,13.340887
9,2025,October,73340.0,68347.0,3.335693e+08,3.454291e+08,5054.048857,0.931920,0.622891,0.662769,-7.643492,-8.251572


In [85]:
# ==========================================
# SAVE ANALYTICAL OUTPUTS
# ==========================================

output_path = project_path / "Output"
output_path.mkdir(exist_ok=True)

monthly_kpi.to_csv(
    output_path / "monthly_workforce_kpis.csv",
    index=False
)

exceptions.to_csv(
    output_path / "workforce_exceptions.csv",
    index=False
)

print("Output files created successfully:")
print("-", output_path / "monthly_workforce_kpis.csv")
print("-", output_path / "workforce_exceptions.csv")

Output files created successfully:
- /Users/munmunshilaka/Documents/Data-Analytics-Portfolio/08_Projects/Project_03_HMRC_Workforce_Cost_Automation/Output/monthly_workforce_kpis.csv
- /Users/munmunshilaka/Documents/Data-Analytics-Portfolio/08_Projects/Project_03_HMRC_Workforce_Cost_Automation/Output/workforce_exceptions.csv
